# Step 1: Check the assumptions with a residual plot and a normal QQ plot

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** `data/processed/model_table.parquet`. **Saves:** `docs/img/step1_assumptions.png`.

The tests in a regression assume that the errors (actual − predicted) have **the same spread** at every demand
level, and are roughly **normally distributed**. The model with every predictor is fitted twice, on raw `pickups`
and on `log(pickups + 1)`, and two plots are checked:

- **Residuals vs fitted.** An even band around 0 is good. A funnel shape means the errors grow (or shrink) with
  demand.
- **Normal QQ plot.** Points on the dashed line mean normally distributed errors. Points bending away at the ends
  mean more extreme errors than normal. Both axes are in standard deviations, so negative values are expected.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
rows = []
for row, (target, label) in enumerate([("pickups", "Raw pickups"), (mu.LOG_TARGET, "log(pickups + 1)")]):
    res = mu.ols(mu.ALL_TERMS, target=target)
    mu.diagnostics(axes[row], res.fittedvalues, res.resid, label)
    rows.append({"target": label, "R2": res.rsquared, "share of negative predictions": (res.fittedvalues < 0).mean()})
    del res
fig.tight_layout()
mu.save_figure(fig, "step1_assumptions.png")
plt.show()
pd.DataFrame(rows).set_index("target")

**Results.**
- **Raw pickups fail both checks.** The residual plot is a funnel (the busier the hour, the bigger the errors), the QQ
  plot bends far away on the right, and 13% of predictions are **negative pickups**.
- **`log(pickups + 1)` is much better.** The band is far more even, almost no predictions are negative (0.01%), and
  the QQ plot is closer to the line. It isn't perfect: quiet hours have somewhat larger errors, and a few hours have
  far less demand than predicted (e.g. holidays).

**Decision:** use `log(pickups + 1)`. The uneven spread that is left is handled by the day-clustered standard errors
used in every test below.